# Uncertainty Inequality — Analysis & Verification

This notebook verifies and compares polynomial constructions that give upper bounds on the constant $C_4$ in the uncertainty inequality
$$A(f)\, A(\widehat{f}) \geq C_4$$
for all even $f$ with $\max(f(0), \widehat{f}(0)) < 0$.

Two approaches are compared:
- **Hermite method** ([Gonçalves et al., 2017](https://www.sciencedirect.com/science/article/pii/S0022247X17301804)): construct $P(x) = \sum_k c_k H_{4k}(x)$ with $P(0) = 0$; the bound is the largest sign-change root of $P(x)/x^2$, squared, divided by $2\pi$.
- **Laguerre LP method** ([Cohn & Gonçalves, 2019](https://arxiv.org/abs/1712.04438)): prescribe double root positions and solve a linear system over generalized Laguerre polynomials; the bound is the largest sign-change root of the quotient, divided by $2\pi$.

Lower is better.

In [1]:
import numpy as np
import sympy
import sys

sys.set_int_max_str_digits(0)
sys.path.insert(0, 'solutions')

## 1. Hermite Method

Core routines for constructing and verifying Hermite polynomial combinations (used by Gonçalves et al. and AlphaEvolve).

In [2]:
def verify_hermite_combination(inputs: sympy.Expr):
    """Check that the Hermite combination vanishes at 0 and is positive at infinity."""
    x = sympy.symbols('x')
    value_at_0 = inputs.subs(x, 0)
    assert value_at_0 == 0, f'The value at 0 is {value_at_0} != 0.'
    assert sympy.limit(inputs, x, sympy.oo) > 0, 'Limit at infty is not positive.'


def find_hermite_combination(coeffs: np.ndarray) -> sympy.Expr:
    """Computes the Hermite combination for given coefficients."""
    m = len(coeffs)
    rational_coeffs = [sympy.Rational(c) for c in coeffs]
    degrees = np.arange(0, 4 * m + 4, 4)
    x = sympy.symbols('x')
    hps = [
        sympy.polys.orthopolys.hermite_poly(n=i, x=x, polys=False)
        for i in degrees
    ]

    partial_fn: sympy.Expr = sum(
        rational_coeffs[i] * hps[i] for i in range(len(rational_coeffs))
    )

    a = hps[-1].subs(x, 0)
    b = -partial_fn.subs(x, 0)
    last_coeff = b / a
    rational_coeffs.append(sympy.Rational(last_coeff))

    res_fn = sum(rational_coeffs[i] * hps[i] for i in range(len(rational_coeffs)))

    if sympy.limit(res_fn, x, sympy.oo) < 0:
        res_fn = -res_fn

    return res_fn


def get_upper_bound(coeffs: np.ndarray) -> float:
    """Computes the upper bound for the given Hermite combination."""
    g_exp = find_hermite_combination(coeffs)
    x = sympy.symbols('x')
    gq_fn = sympy.exquo(g_exp, x**2)
    rroots = sympy.real_roots(gq_fn, x)
    approx_roots = list()
    largest_sign_change = 0

    for root in rroots:
        approx_root = root.eval_rational(n=200)
        approx_root_p = approx_root + sympy.Rational(1e-198)
        approx_root_m = approx_root - sympy.Rational(1e-198)
        approx_roots.append(approx_root)
        is_sign_change = (
            (gq_fn.subs(x, approx_root_p) > 0 and gq_fn.subs(x, approx_root_m) < 0)
            or (
                gq_fn.subs(x, approx_root_p) < 0
                and gq_fn.subs(x, approx_root_m) > 0
            )
        )
        if is_sign_change:
            largest_sign_change = max(largest_sign_change, approx_root)

    return float(largest_sign_change**2) / (2 * np.pi)

## 2. Laguerre LP Method

The Laguerre LP framework ([Cohn & Gonçalves, arXiv:1712.04438](https://arxiv.org/abs/1712.04438)) prescribes double root positions $z_1, \ldots, z_k > 0$ and constructs $g(x)$ as a linear combination of generalized Laguerre polynomials ($\alpha = -1/2$, even degrees) with:
1. $g(0) = 0$, $g'(0) = 1$
2. $g(z_i) = g'(z_i) = 0$ (double roots)

The upper bound is the largest sign-change root of $g(x) / (x \prod_i (x - z_i)^2)$, divided by $2\pi$.

In [3]:
def _find_laguerre_combination(zs):
    """Construct Laguerre test function with prescribed double roots."""
    m = len(zs)
    alpha = sympy.Rational(1, 2) - 1
    x = sympy.symbols('x')
    degrees = np.arange(0, 4 * m + 4, 2)
    lps = [
        sympy.polys.orthopolys.laguerre_poly(n=int(i), x=x, alpha=alpha, polys=False)
        for i in degrees
    ]
    num_lps = len(lps)
    num_conditions = 2 * m + 2

    mat = sympy.Matrix(num_conditions, num_lps, lambda i, j: 0)
    b = sympy.Matrix(num_conditions, 1, lambda i, j: 0)
    b[1] = 1

    for j in range(num_lps):
        mat[0, j] = lps[j].subs(x, 0)
        mat[1, j] = lps[j].diff(x).subs(x, 0)

    for i in range(m):
        zi = sympy.Rational(zs[i])
        for j in range(num_lps):
            mat[2 * i + 2, j] = lps[j].subs(x, zi)
            mat[2 * i + 3, j] = lps[j].diff(x).subs(x, zi)

    coeffs = mat.LUsolve(b)
    return sum(coeffs[i] * lps[i] for i in range(num_lps))


def get_upper_bound_laguerre(zs) -> float:
    """Compute upper bound using the Laguerre LP framework.

    Matches the Einstein Arena verifier (uncertainty-principle.ts).
    """
    g_fn = _find_laguerre_combination(zs)
    x = sympy.symbols('x')

    div = sympy.prod([(x - sympy.Rational(z)) ** 2 for z in zs]) * x
    gq_fn = sympy.exquo(g_fn, div)

    real_roots = sympy.real_roots(gq_fn, x)
    gq_np = sympy.lambdify(x, gq_fn, modules="numpy")
    largest_sign_change = 0.0
    for root in real_roots:
        r_val = float(root.evalf(30))
        eps = 1e-6
        if np.sign(gq_np(r_val - eps)) != np.sign(gq_np(r_val + eps)):
            largest_sign_change = max(largest_sign_change, r_val)

    assert largest_sign_change > 0, "No sign-changing roots found."
    return float(largest_sign_change) / (2 * np.pi)

## 3. Load Solutions

In [4]:
from goncalves_2017 import coefficients as coefficients_goncalves
from alphaevolve_2025 import coefficients as coefficients_alphaevolve
from alphaevolve2_2025 import laguerre_double_roots as roots_alphaevolve_v2
from togetherai_2026 import laguerre_double_roots as roots_togetherai

hermite_solutions = [
    ("Gonçalves et al. (2017)", coefficients_goncalves),
    ("AlphaEvolve (2025)", coefficients_alphaevolve),
]

laguerre_solutions = [
    ("AlphaEvolveV2 (2026)", roots_alphaevolve_v2),
    ("Together AI (2026)", roots_togetherai),
]

for name, coeffs in hermite_solutions:
    print(f'{name}: {len(coeffs)} Hermite coefficients')
for name, roots in laguerre_solutions:
    print(f'{name}: {len(roots)} Laguerre double roots')

Gonçalves et al. (2017): 3 Hermite coefficients
AlphaEvolve (2025): 3 Hermite coefficients
AlphaEvolveV2 (2026): 6 Laguerre double roots
Together AI (2026): 13 Laguerre double roots


## 4. Verification

Verify structural properties of each construction:
- **Hermite:** $P(0) = 0$ and $P(x) \to +\infty$ as $x \to \infty$
- **Laguerre LP:** $g(0) = 0$, $g'(0) = 1$, and double roots at prescribed positions

In [5]:
print("Hermite method:")
for name, coeffs in hermite_solutions:
    expr = find_hermite_combination(coeffs)
    verify_hermite_combination(expr)
    print(f'  {name}: PASS (P(0)=0, positive at infinity)')

print("\nLaguerre LP method:")
for name, roots in laguerre_solutions:
    g_fn = _find_laguerre_combination(roots.tolist())
    x = sympy.symbols('x')
    assert g_fn.subs(x, 0) == 0, "g(0) != 0"
    assert g_fn.diff(x).subs(x, 0) == 1, "g'(0) != 1"
    for z in roots:
        zi = sympy.Rational(float(z))
        assert g_fn.subs(x, zi) == 0, f"g({z}) != 0"
    print(f'  {name}: PASS (g(0)=0, g\'(0)=1, double roots verified)')

print('\nAll constructions pass verification.')

Hermite method:
  Gonçalves et al. (2017): PASS (P(0)=0, positive at infinity)
  AlphaEvolve (2025): PASS (P(0)=0, positive at infinity)

Laguerre LP method:
  AlphaEvolveV2 (2026): PASS (g(0)=0, g'(0)=1, double roots verified)
  Together AI (2026): PASS (g(0)=0, g'(0)=1, double roots verified)

All constructions pass verification.


## 5. Compute Upper Bounds

In [6]:
print("=" * 70)
print("UPPER BOUNDS")
print("=" * 70)
print()

scores = {}

for name, coeffs in hermite_solutions:
    bound = get_upper_bound(coeffs)
    scores[name] = bound
    print(f'{name} [Hermite]')
    print(f'  Parameters: {len(coeffs)} coefficients')
    print(f'  Upper bound: C_4 <= {bound:.6f}')
    print()

for name, roots in laguerre_solutions:
    bound = get_upper_bound_laguerre(roots.tolist())
    scores[name] = bound
    print(f'{name} [Laguerre LP]')
    print(f'  Parameters: {len(roots)} double roots')
    print(f'  Upper bound: C_4 <= {bound:.6f}')
    print()

print("=" * 70)
print("COMPARISON TABLE")
print("=" * 70)
print(f"{'Method':<30} {'Upper Bound':>18}")
print("-" * 50)
for name in scores:
    marker = " <-- best" if scores[name] == min(scores.values()) else ""
    print(f"{name:<30} {scores[name]:>18.6f}{marker}")

UPPER BOUNDS

Gonçalves et al. (2017) [Hermite]
  Parameters: 3 coefficients
  Upper bound: C_4 <= 0.352296

AlphaEvolve (2025) [Hermite]
  Parameters: 3 coefficients
  Upper bound: C_4 <= 0.352099

AlphaEvolveV2 (2026) [Laguerre LP]
  Parameters: 6 double roots
  Upper bound: C_4 <= 0.328271

Together AI (2026) [Laguerre LP]
  Parameters: 13 double roots
  Upper bound: C_4 <= 0.318855

COMPARISON TABLE
Method                                Upper Bound
--------------------------------------------------
Gonçalves et al. (2017)                  0.352296
AlphaEvolve (2025)                       0.352099
AlphaEvolveV2 (2026)                     0.328271
Together AI (2026)                       0.318855 <-- best
